# 딥소각 얼굴가드 Colab 실행

이 노트북은 환경 점검, 자원 추정, manifest 누수 검사를 실행한다. 얼굴 원본·정렬본·임베딩은 생체정보이다. 제공 약관의 클라우드/국외 반출 허용과 프로젝트 승인이 모두 없으면 hosted Colab/Drive에 올리지 말고 local runtime을 사용하세요.

공유 전에 **Edit → Clear all outputs**를 실행하고, secret·실제 파일경로·subject ID를 cell에 적지 않습니다.

In [ ]:
#@title 1. 실행 모드와 보안 게이트
REPO_URL = 'https://github.com/Chunbae-A/face-image.git' #@param {type:'string'}
BRANCH = 'feat/faceguard-experiment-plan' #@param {type:'string'}
HOSTED_CLOUD_DATA_APPROVED = False #@param {type:'boolean'}
USE_GOOGLE_DRIVE = False #@param {type:'boolean'}

if USE_GOOGLE_DRIVE and not HOSTED_CLOUD_DATA_APPROVED:
    raise PermissionError(
        'Drive mount blocked: cloud/foreign transfer approval is required.'
    )
print({
    'branch': BRANCH,
    'hosted_cloud_data_approved': HOSTED_CLOUD_DATA_APPROVED,
    'use_google_drive': USE_GOOGLE_DRIVE,
})

In [ ]:
# 2. 코드 가져오기: 작업 브랜치를 push한 후 실행
from pathlib import Path
import os
import subprocess

repo_dir = Path('/content/face-image')
if not repo_dir.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(repo_dir)],
        check=True,
    )
os.chdir(repo_dir)
print(subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
# 3. 할당된 하드웨어/디스크 실측 (매 세션 재실행)
import json
import subprocess
import torch

inventory = json.loads(subprocess.check_output(
    ['python', 'scripts/faceguard_plan.py', 'env', '--path', '.'],
    text=True,
))
safe_inventory = {
    key: value for key, value in inventory.items()
    if key not in {'path'}
}
print(json.dumps(safe_inventory, indent=2, ensure_ascii=False))
if torch.cuda.is_available():
    print('CUDA:', torch.cuda.get_device_name(0))
    print('VRAM bytes:', torch.cuda.get_device_properties(0).total_memory)
else:
    print('WARNING: GPU runtime is not active.')

In [ ]:
# 4. 승인된 경우에만 Drive mount
drive_root = None
if USE_GOOGLE_DRIVE:
    if not HOSTED_CLOUD_DATA_APPROVED:
        raise PermissionError('Cloud data approval is required.')
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    drive_root = Path('/content/drive/MyDrive/face-image-secure')
    drive_root.mkdir(parents=True, exist_ok=True)
    print('Approved persistent root is configured.')
else:
    print('Drive is not mounted. Use synthetic/publicly permitted debug data only.')

In [ ]:
# 5. 100GB 다운로드/저장 추정값 재계산
print(subprocess.check_output([
    'python', 'scripts/faceguard_plan.py', 'download',
    '--size-gb', '100', '--efficiency', '0.8',
], text=True))
print(subprocess.check_output([
    'python', 'scripts/faceguard_plan.py', 'storage',
    '--compressed-gb', '100', '--unpacked-gb', '150',
    '--preprocessed-gb', '30', '--images', '1000000',
], text=True))

In [ ]:
# 6. 스키마/누수 검사: 예제 manifest로 먼저 실행
result = subprocess.run([
    'python', 'scripts/validate_faceguard_manifest.py',
    'examples/faceguard_manifest.csv',
], check=False, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    raise RuntimeError('Manifest leakage validation failed.')

## 다음 실행 순서

1. 승인된 Debug shard(20~50명)로 decode·hash·얼굴 탐지·5-point 정렬·resume를 검증합니다.
2. manifest validator가 0 error일 때만 ArcFace baseline을 실행합니다.
3. 세션이 종료될 수 있으므로 shard별 checkpoint, `_SUCCESS`, model/config/data hash를 영구 저장소에 기록합니다.
4. Debug 처리량으로 Pilot 시간/비용을 추정한 후 200~500명을 실행합니다.
5. Pilot Go 판정 없이 100GB Full Set을 복사하지 않습니다.